# 03 — Score Sentiment & Combine

Scores every unscored headline in `news_articles` with FinBERT, aggregates scores into `daily_sentiment`, then matches sentiment to the next trading day and merges with `stock_prices` as a quick sanity check that sentiment actually correlates with price movement before we train anything.

**Note:** fine-tuning FinBERT on Indian financial news is out of scope for this rebuild phase — `utils/finbert.py` automatically falls back to base `ProsusAI/finbert` since no fine-tuned weights are present. Swapping in a fine-tuned model later needs zero changes here.

Run this after `02_fetch_news.ipynb`. First run will download FinBERT (~440MB) — subsequent runs are cached.

In [1]:
!pip install "numpy==1.26.4" scipy scikit-learn transformers torch -q

## Load unscored articles

In [2]:
import sys
sys.path.append('..')
import pandas as pd
from utils.db import get_connection

conn = get_connection()
df_news = pd.read_sql("SELECT * FROM news_articles", conn)
conn.close()

print(f"Total articles : {len(df_news):,}")
print(f"Already scored : {df_news['sentiment_label'].notna().sum():,}")
print(f"Unscored       : {df_news['sentiment_label'].isna().sum():,}")

C:\Users\acer\AppData\Local\Temp\ipykernel_22656\3065401656.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_news = pd.read_sql("SELECT * FROM news_articles", conn)


Total articles : 51,523
Already scored : 0
Unscored       : 51,523


## Sanity check FinBERT on a few known headlines

In [3]:
from utils.finbert import score_sentiment

tests = [
    "Reliance Q4 profit rises 40 percent beating all estimates",
    "Sensex crashes 1000 points as FII selling intensifies",
    "TCS misses Q3 estimates as attrition rises to 20 percent",
    "Infosys cuts annual revenue guidance amid weak demand",
]
print(f"{'TEXT':<58} {'LABEL':<10} {'COMPOUND'}")
print("-" * 80)
for text in tests:
    label, score, compound = score_sentiment(text)
    print(f"{text[:57]:<58} {label:<10} {compound:+.4f}")

Fine-tuned weights not found — loading base FinBERT (ProsusAI/finbert)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FinBERT ready.
TEXT                                                       LABEL      COMPOUND
--------------------------------------------------------------------------------
Reliance Q4 profit rises 40 percent beating all estimates  positive   +0.9161
Sensex crashes 1000 points as FII selling intensifies      negative   -0.4781
TCS misses Q3 estimates as attrition rises to 20 percent   negative   -0.8608
Infosys cuts annual revenue guidance amid weak demand      negative   -0.9629


## Score all unscored articles in batches

In [4]:
from utils.finbert import score_batch

unscored = df_news[df_news['sentiment_label'].isna()].copy()
print(f"Articles to score : {len(unscored):,}")

if len(unscored) > 0:
    texts = unscored['content'].tolist()
    results = score_batch(texts, batch_size=32)

    unscored['sentiment_label'] = [r[0] for r in results]
    unscored['sentiment_score'] = [r[1] for r in results]
    unscored['sentiment_compound'] = [r[2] for r in results]

    print(f"\nDone.")
    print(unscored['sentiment_label'].value_counts())
else:
    print("Nothing to score.")

Articles to score : 51,523
  Scored 32 / 51,523
  Scored 64 / 51,523
  Scored 96 / 51,523
  Scored 128 / 51,523
  Scored 160 / 51,523
  Scored 192 / 51,523
  Scored 224 / 51,523
  Scored 256 / 51,523
  Scored 288 / 51,523
  Scored 320 / 51,523
  Scored 352 / 51,523
  Scored 384 / 51,523
  Scored 416 / 51,523
  Scored 448 / 51,523
  Scored 480 / 51,523
  Scored 512 / 51,523
  Scored 544 / 51,523
  Scored 576 / 51,523
  Scored 608 / 51,523
  Scored 640 / 51,523
  Scored 672 / 51,523
  Scored 704 / 51,523
  Scored 736 / 51,523
  Scored 768 / 51,523
  Scored 800 / 51,523
  Scored 832 / 51,523
  Scored 864 / 51,523
  Scored 896 / 51,523
  Scored 928 / 51,523
  Scored 960 / 51,523
  Scored 992 / 51,523
  Scored 1,024 / 51,523
  Scored 1,056 / 51,523
  Scored 1,088 / 51,523
  Scored 1,120 / 51,523
  Scored 1,152 / 51,523
  Scored 1,184 / 51,523
  Scored 1,216 / 51,523
  Scored 1,248 / 51,523
  Scored 1,280 / 51,523
  Scored 1,312 / 51,523
  Scored 1,344 / 51,523
  Scored 1,376 / 51,523
  Scor

## Save scores back to news_articles

In [6]:
from psycopg2.extras import execute_values
from utils.db import get_connection

conn = get_connection()
cur = conn.cursor()

update_data = [
    (int(row['id']), row['sentiment_label'], row['sentiment_score'], row['sentiment_compound'])
    for _, row in unscored.iterrows()
]

BATCH_SIZE = 1000
for i in range(0, len(update_data), BATCH_SIZE):
    batch = update_data[i:i + BATCH_SIZE]
    execute_values(cur, """
        UPDATE news_articles AS n
        SET sentiment_label    = v.label,
            sentiment_score    = v.score,
            sentiment_compound = v.compound
        FROM (VALUES %s) AS v(id, label, score, compound)
        WHERE n.id = v.id
    """, batch, template="(%s, %s, %s, %s)")
    conn.commit()
    print(f"  Saved {min(i + BATCH_SIZE, len(update_data)):,} / {len(update_data):,}")

cur.close()
conn.close()
print("All scores saved.")

  Saved 1,000 / 51,523
  Saved 2,000 / 51,523
  Saved 3,000 / 51,523
  Saved 4,000 / 51,523
  Saved 5,000 / 51,523
  Saved 6,000 / 51,523
  Saved 7,000 / 51,523
  Saved 8,000 / 51,523
  Saved 9,000 / 51,523
  Saved 10,000 / 51,523
  Saved 11,000 / 51,523
  Saved 12,000 / 51,523
  Saved 13,000 / 51,523
  Saved 14,000 / 51,523
  Saved 15,000 / 51,523
  Saved 16,000 / 51,523
  Saved 17,000 / 51,523
  Saved 18,000 / 51,523
  Saved 19,000 / 51,523
  Saved 20,000 / 51,523
  Saved 21,000 / 51,523
  Saved 22,000 / 51,523
  Saved 23,000 / 51,523
  Saved 24,000 / 51,523
  Saved 25,000 / 51,523
  Saved 26,000 / 51,523
  Saved 27,000 / 51,523
  Saved 28,000 / 51,523
  Saved 29,000 / 51,523
  Saved 30,000 / 51,523
  Saved 31,000 / 51,523
  Saved 32,000 / 51,523
  Saved 33,000 / 51,523
  Saved 34,000 / 51,523
  Saved 35,000 / 51,523
  Saved 36,000 / 51,523
  Saved 37,000 / 51,523
  Saved 38,000 / 51,523
  Saved 39,000 / 51,523
  Saved 40,000 / 51,523
  Saved 41,000 / 51,523
  Saved 42,000 / 51,523
 

## Aggregate into daily_sentiment

In [7]:
conn = get_connection()
df_news = pd.read_sql("SELECT * FROM news_articles", conn)
conn.close()

df_news['date'] = pd.to_datetime(df_news['published_at']).dt.date

daily = df_news.groupby(['ticker', 'date']).agg(
    article_count=('sentiment_compound', 'count'),
    avg_compound=('sentiment_compound', 'mean'),
    max_compound=('sentiment_compound', 'max'),
    min_compound=('sentiment_compound', 'min'),
    std_compound=('sentiment_compound', 'std'),
    positive_count=('sentiment_label', lambda x: (x == 'positive').sum()),
    negative_count=('sentiment_label', lambda x: (x == 'negative').sum()),
).reset_index()

daily['std_compound'] = daily['std_compound'].fillna(0)
daily['date'] = pd.to_datetime(daily['date'])

print(f"Daily sentiment rows : {len(daily):,}")
print(f"Tickers covered      : {daily['ticker'].nunique()}")
print(daily.head(5).to_string())

C:\Users\acer\AppData\Local\Temp\ipykernel_22656\1738810587.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_news = pd.read_sql("SELECT * FROM news_articles", conn)


Daily sentiment rows : 27,598
Tickers covered      : 50
        ticker       date  article_count  avg_compound  max_compound  min_compound  std_compound  positive_count  negative_count
0  ADANIENT.NS 2016-11-27              1      -0.04530       -0.0453       -0.0453      0.000000               0               0
1  ADANIENT.NS 2017-01-05              1       0.91080        0.9108        0.9108      0.000000               1               0
2  ADANIENT.NS 2017-02-14              2       0.88895        0.8946        0.8833      0.007990               2               0
3  ADANIENT.NS 2017-02-15              1      -0.93130       -0.9313       -0.9313      0.000000               0               1
4  ADANIENT.NS 2017-02-17              2      -0.13745       -0.1131       -0.1618      0.034436               0               0


## Save to daily_sentiment (safe to re-run — ON CONFLICT DO NOTHING)

In [9]:
from psycopg2.extras import execute_values
from utils.db import get_connection

conn = get_connection()
cur = conn.cursor()

rows = [
    (
        row['ticker'], row['date'],
        int(row['article_count']),
        float(row['avg_compound']),
        float(row['max_compound']),
        float(row['min_compound']),
        float(row['std_compound']),
        int(row['positive_count']),
        int(row['negative_count'])
    )
    for _, row in daily.iterrows()
]

BATCH_SIZE = 1000
for i in range(0, len(rows), BATCH_SIZE):
    batch = rows[i:i + BATCH_SIZE]
    execute_values(cur, """
        INSERT INTO daily_sentiment
            (ticker, date, article_count, avg_compound, max_compound,
             min_compound, std_compound, positive_count, negative_count)
        VALUES %s
        ON CONFLICT (ticker, date) DO NOTHING
    """, batch)
    conn.commit()
    print(f"  Saved {min(i + BATCH_SIZE, len(rows)):,} / {len(rows):,}")

cur.close()
conn.close()
print(f"\nTotal saved : {len(rows):,} rows to daily_sentiment.")

  Saved 1,000 / 27,598
  Saved 2,000 / 27,598
  Saved 3,000 / 27,598
  Saved 4,000 / 27,598
  Saved 5,000 / 27,598
  Saved 6,000 / 27,598
  Saved 7,000 / 27,598
  Saved 8,000 / 27,598
  Saved 9,000 / 27,598
  Saved 10,000 / 27,598
  Saved 11,000 / 27,598
  Saved 12,000 / 27,598
  Saved 13,000 / 27,598
  Saved 14,000 / 27,598
  Saved 15,000 / 27,598
  Saved 16,000 / 27,598
  Saved 17,000 / 27,598
  Saved 18,000 / 27,598
  Saved 19,000 / 27,598
  Saved 20,000 / 27,598
  Saved 21,000 / 27,598
  Saved 22,000 / 27,598
  Saved 23,000 / 27,598
  Saved 24,000 / 27,598
  Saved 25,000 / 27,598
  Saved 26,000 / 27,598
  Saved 27,000 / 27,598
  Saved 27,598 / 27,598

Total saved : 27,598 rows to daily_sentiment.


## Combine with prices — match sentiment to next trading day

News published on a non-trading day (weekend/holiday) gets matched forward to the next trading day within a 3-day window, then merged with `stock_prices` purely as a signal-strength sanity check (this merge is *not* saved anywhere — the real feature engineering for modeling happens fresh in notebook 04).

In [10]:
conn = get_connection()
df_stock = pd.read_sql("SELECT * FROM stock_prices", conn)
conn.close()

df_stock['date'] = pd.to_datetime(df_stock['date'])
daily['date'] = pd.to_datetime(daily['date'])

trading_days = df_stock[['ticker', 'date']].copy()

def match_to_next_trading_day(sent_df, stock_df):
    results = []
    for _, row in sent_df.iterrows():
        ticker = row['ticker']
        news_date = row['date']
        ticker_days = stock_df[stock_df['ticker'] == ticker]['date']
        window = ticker_days[
            (ticker_days > news_date) &
            (ticker_days <= news_date + pd.Timedelta(days=3))
        ]
        if len(window) > 0:
            results.append({
                'ticker': ticker,
                'matched_date': window.min(),
                'avg_compound': row['avg_compound'],
                'article_count': row['article_count'],
            })
    return pd.DataFrame(results)

print("Matching news to next trading day...")
matched = match_to_next_trading_day(daily, trading_days)
print(f"Original rows : {len(daily):,}")
print(f"Matched rows  : {len(matched):,}")

C:\Users\acer\AppData\Local\Temp\ipykernel_22656\357129507.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_stock = pd.read_sql("SELECT * FROM stock_prices", conn)


Matching news to next trading day...
Original rows : 27,598
Matched rows  : 26,955


In [16]:
merged = pd.merge(
    df_stock,
    matched.rename(columns={'matched_date': 'date'}),
    on=['ticker', 'date'],
    how='inner'
)
merged = merged.sort_values(['ticker', 'date']).reset_index(drop=True)

merged['future_close_1d'] = merged.groupby('ticker')['close_price'].shift(-1)
merged['target_1d'] = (merged['future_close_1d'] > merged['close_price']).astype(int)
merged = merged.dropna(subset=['future_close_1d'])

pos_1d = merged[merged['avg_compound'] > 0.1]['target_1d'].mean() * 100
neg_1d = merged[merged['avg_compound'] < -0.1]['target_1d'].mean() * 100

print("=== SENTIMENT SIGNAL CHECK (1 day ahead) ===")
print(f"Positive news → price UP next day : {pos_1d:.1f}% of the time")
print(f"Negative news → price UP next day : {neg_1d:.1f}% of the time")
print(f"Gap (higher = stronger signal)    : {pos_1d - neg_1d:.1f} points")

=== SENTIMENT SIGNAL CHECK (1 day ahead) ===
Positive news → price UP next day : 48.2% of the time
Negative news → price UP next day : 47.7% of the time
Gap (higher = stronger signal)    : 0.5 points
